In [ ]:
import sys
import os
from pathlib import Path

if 'google.colab' in sys.modules:
    print("Ambiente Colab rilevato. Inizializzazione in corso...")
    
    # 1. Monta Drive (se non già montato)
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
        
    # Imposta la root del Drive su Colab
    BASE_DIR = Path('/content/drive/Shareddrives/Progetti/Progetto_MLDM')

    GIT_BRANCH = 'zone-test'
    
    # 2. Clona/Aggiorna la repo
    REPO_DIR = '/content/crop-spatial-classification'
    if not os.path.exists(REPO_DIR):
        !git clone -b {GIT_BRANCH} https://github.com/SimoRinaldi/crop-spatial-classification.git {REPO_DIR}
    else:
        !cd {REPO_DIR} && git fetch origin && git checkout -B {GIT_BRANCH} origin/{GIT_BRANCH} && git reset --hard origin/{GIT_BRANCH} && git clean -fd
    
    # 3. Aggancia la cartella per permettere gli import
    if REPO_DIR not in sys.path:
        sys.path.append(REPO_DIR)
        
    # 4. Installa i requisiti in modo silenzioso
    %pip install -q -r {REPO_DIR}/requirements.txt
    print("Setup ambiente Colab completato! Branch attivo su Colab: ", GIT_BRANCH)
    
else:
    print("Ambiente Locale rilevato. Procedo con l'esecuzione...")
    
    BASE_DIR = Path.cwd().parent

    if str(BASE_DIR) not in sys.path:
        sys.path.append(str(BASE_DIR))

# =============================
# DEFINIZIONE PATH UNIVERSALI
# =============================
DATA_DIR = BASE_DIR / 'data'
RAW_DIR = DATA_DIR / 'raw'
INTERIM_DIR = DATA_DIR / 'interim'
PROCESSED_DIR = DATA_DIR / 'processed'
MODELS_DIR = DATA_DIR / 'models'

# Test di verifica per assicurarsi che i dati siano accessibili
if RAW_DIR.exists():
    print(f"✅ Collegamento ai dati riuscito! Cartella raw: {RAW_DIR}")
else:
    print(f"❌ Attenzione: Cartella non trovata in {RAW_DIR}. Verifica il nome, il mount o il path locale.")
     


Ambiente Colab rilevato. Inizializzazione in corso...
branch 'zone-test' set up to track 'origin/zone-test'.
Reset branch 'zone-test'
Your branch is up to date with 'origin/zone-test'.
HEAD is now at c266982 refactor(test-dataset): implement 100k random sampling on test zones at 10m resolution
Setup ambiente Colab completato! Branch attivo su Colab:  zone-test
✅ Collegamento ai dati riuscito! Cartella raw: /content/drive/Shareddrives/Progetti/Progetto_MLDM/data/raw


# 01 - Estrazione del ground truth e campionamento delle colture
estrae un campione bilanciato di punti per ciascun tipo di coltura e genera la legenda associata.

#### Ricerca dei file TIF per ciascun anno

In [2]:
from pathlib import Path

main_directory = Path(f"{DATA_DIR}/raw/zone_test")
tifs_3035 = {}

for item in sorted(main_directory.iterdir()):
    if item.is_dir():  
        zone_dir = item
        year_dir = sorted(zone_dir.iterdir()).pop()
        if year_dir.is_dir():
            # trova tutti i .tif per questa zona per il primo anno che trova
            tifs_3035[zone_dir.name] = {
                "tif": [str(tif) for tif in year_dir.rglob("*.tif")],
                "xml": [str(xml) for xml in year_dir.rglob("*.xml")]
            }
            print(f"Zona {zone_dir.name}: trovati {len(tifs_3035[zone_dir.name])} file .tif per l'anno {year_dir.name}")
        else:
            print(f"Cartella zona {zone_dir.name} non trovata in {main_directory}")

print(tifs_3035)

Zona albacete_spagna: trovati 2 file .tif per l'anno 2024
Zona jerez_spagna: trovati 2 file .tif per l'anno 2024
Zona rovato: trovati 2 file .tif per l'anno 2024
Zona vicino_capitanata: trovati 2 file .tif per l'anno 2024
{'albacete_spagna': {'tif': ['/content/drive/Shareddrives/Progetti/Progetto_MLDM/data/raw/zone_test/albacete_spagna/2024/CLMS_HRLVLCC_CTY_S2024_R10m_E32N18_03035_V01_R00/CLMS_HRLVLCC_CTY_S2024_R10m_E32N18_03035_V01_R00.tif'], 'xml': ['/content/drive/Shareddrives/Progetti/Progetto_MLDM/data/raw/zone_test/albacete_spagna/2024/CLMS_HRLVLCC_CTY_S2024_R10m_E32N18_03035_V01_R00/CLMS_HRLVLCC_CTY_S2024_R10m_E32N18_03035_V01_R00.tif.aux.xml', '/content/drive/Shareddrives/Progetti/Progetto_MLDM/data/raw/zone_test/albacete_spagna/2024/CLMS_HRLVLCC_CTY_S2024_R10m_E32N18_03035_V01_R00/CLMS_HRLVLCC_CTY_S2024_R10m_E32N18_03035_V01_R00.xml']}, 'jerez_spagna': {'tif': ['/content/drive/Shareddrives/Progetti/Progetto_MLDM/data/raw/zone_test/jerez_spagna/2024/CLMS_HRLVLCC_CTY_S2024_R10m_

#### Riproiezione in coordinate GPS (EPSG:4326)

In [3]:
from pathlib import Path
import rioxarray
from rasterio.enums import Resampling

tifs_4326 = {}

for zone_name, file_paths in tifs_3035.items():
    zone_file_list = []

    for xml_files in file_paths["tif"]:
        # costruisce il percorso del file riproiettato in EPSG:4326
        file_name = xml_files.replace("03035", "4326").replace("raw", "processed")
        file_path = Path(file_name)
        
        # se il file riproiettato esiste già, salta la riproiezione
        if file_path.is_file():
            zone_file_list.append(file_name)
            continue
        
        # crea la cartella di destinazione se non esiste
        file_path.parent.mkdir(parents=True, exist_ok=True)
        
        # apre il file GeoTIFF originale (EPSG:3035)
        raster_3035 = rioxarray.open_rasterio(xml_files)

        # riproiezione in EPSG:4326, usando NEAREST per preservare i codici delle colture
        raster_4326 = raster_3035.rio.reproject("EPSG:4326", resampling=Resampling.nearest)

        # salva il file riproiettato
        raster_4326.rio.to_raster(file_name)
        raster_3035.close()

        zone_file_list.append(file_name)
        
    tifs_4326[zone_name] = zone_file_list

print(f"\nRiproiezione completata per {sum(len(v) for v in tifs_4326.values())} file")


Riproiezione completata per 4 file


#### Creazione e salvataggio legenda delle colture

In [4]:
import pandas as pd
import xml.etree.ElementTree as ET
from src.config import EXCLUDED_CROP_CLASSES

for zone_name, file_paths in tifs_3035.items():
    zone_file_list = []

    for bbox in file_paths["xml"]:
        if ".aux" not in bbox:
            continue

        # prende il primo file trovato
        file_path = bbox
        print(f"Trovato file .aux.xml: {file_path}")

        tree = ET.parse(file_path)
        root = tree.getroot()

        summary = {}

        # estrae il codice e il nome della coltura
        for r in root.findall(".//Row"):
            fields = r.findall("F")
            
            crop_code = int(fields[0].text)     

            if crop_code in EXCLUDED_CROP_CLASSES:
                continue
            
            crop_count = fields[1].text
            crop_name = fields[2].text

            summary[crop_code] = {"name": crop_name, "count": crop_count}

        df_summary = (
            pd.DataFrame.from_dict(summary, orient="index")
            .rename_axis("crop_code")
            .reset_index()
        )

        print(f"Zona: {zone_name}")
        display(df_summary)

Trovato file .aux.xml: /content/drive/Shareddrives/Progetti/Progetto_MLDM/data/raw/zone_test/albacete_spagna/2024/CLMS_HRLVLCC_CTY_S2024_R10m_E32N18_03035_V01_R00/CLMS_HRLVLCC_CTY_S2024_R10m_E32N18_03035_V01_R00.tif.aux.xml
Zona: albacete_spagna


,crop_code,name,count
0,1110,Wheat,3556780
1,1120,Barley,10572503
2,1130,Maize,866668
3,1140,Rice,0
4,1150,Other Cereals,7383710
5,1210,Fresh Vegetables,2017232
6,1220,Dry Pulses,2750707
7,1310,Potatoes,90132
8,1320,Sugar Beet,25
9,1410,Sunflower,198735


Trovato file .aux.xml: /content/drive/Shareddrives/Progetti/Progetto_MLDM/data/raw/zone_test/jerez_spagna/2024/CLMS_HRLVLCC_CTY_S2024_R10m_E28N16_03035_V01_R00/CLMS_HRLVLCC_CTY_S2024_R10m_E28N16_03035_V01_R00.tif.aux.xml
Zona: jerez_spagna


,crop_code,name,count
0,1110,Wheat,3614311
1,1120,Barley,567641
2,1130,Maize,260408
3,1140,Rice,87895
4,1150,Other Cereals,1894213
5,1210,Fresh Vegetables,1611515
6,1220,Dry Pulses,1075100
7,1310,Potatoes,128206
8,1320,Sugar Beet,0
9,1410,Sunflower,2833350


Trovato file .aux.xml: /content/drive/Shareddrives/Progetti/Progetto_MLDM/data/raw/zone_test/rovato/2024/CLMS_HRLVLCC_CTY_S2024_R10m_E43N24_03035_V01_R00/CLMS_HRLVLCC_CTY_S2024_R10m_E43N24_03035_V01_R00.tif.aux.xml
Zona: rovato


,crop_code,name,count
0,1110,Wheat,8670783
1,1120,Barley,3002656
2,1130,Maize,23153475
3,1140,Rice,328171
4,1150,Other Cereals,225989
5,1210,Fresh Vegetables,3158189
6,1220,Dry Pulses,283663
7,1310,Potatoes,72002
8,1320,Sugar Beet,47437
9,1410,Sunflower,332955


Trovato file .aux.xml: /content/drive/Shareddrives/Progetti/Progetto_MLDM/data/raw/zone_test/vicino_capitanata/2024/CLMS_HRLVLCC_CTY_S2024_R10m_E47N21_03035_V01_R00/CLMS_HRLVLCC_CTY_S2024_R10m_E47N21_03035_V01_R00.tif.aux.xml
Zona: vicino_capitanata


,crop_code,name,count
0,1110,Wheat,1602002
1,1120,Barley,670789
2,1130,Maize,12332
3,1140,Rice,188
4,1150,Other Cereals,39137
5,1210,Fresh Vegetables,494150
6,1220,Dry Pulses,531316
7,1310,Potatoes,10912
8,1320,Sugar Beet,0
9,1410,Sunflower,340336


In [5]:
import xml.etree.ElementTree as ET

for zone_name, file_paths in tifs_3035.items():
    if not isinstance(file_paths, dict) or "xml" not in file_paths:
        continue

    for bbox in file_paths["xml"]:
        if ".aux" in str(bbox):
            continue

        tree = ET.parse(bbox)
        root = tree.getroot()

        bbox_node = root.find(".//{*}EX_GeographicBoundingBox")

        if bbox_node is not None:
            bbox = [
                float(bbox_node.find("{*}westBoundLongitude/{*}Decimal").text),
                float(bbox_node.find("{*}southBoundLatitude/{*}Decimal").text),
                float(bbox_node.find("{*}eastBoundLongitude/{*}Decimal").text),
                float(bbox_node.find("{*}northBoundLatitude/{*}Decimal").text),
            ]
            print(f"Zona: {zone_name} -> BBox: {bbox}")
            file_paths["bbox"] = bbox

print(tifs_3035)


Zona: albacete_spagna -> BBox: [-2.9952, 38.4339, -1.6912, 39.4732]
Zona: jerez_spagna -> BBox: [-7.065, 35.955, -5.7666, 37.0341]
Zona: rovato -> BBox: [9.7312, 44.7065, 11.0113, 45.6125]
Zona: vicino_capitanata -> BBox: [14.5536, 41.8387, 15.8382, 42.8017]
{'albacete_spagna': {'tif': ['/content/drive/Shareddrives/Progetti/Progetto_MLDM/data/raw/zone_test/albacete_spagna/2024/CLMS_HRLVLCC_CTY_S2024_R10m_E32N18_03035_V01_R00/CLMS_HRLVLCC_CTY_S2024_R10m_E32N18_03035_V01_R00.tif'], 'xml': ['/content/drive/Shareddrives/Progetti/Progetto_MLDM/data/raw/zone_test/albacete_spagna/2024/CLMS_HRLVLCC_CTY_S2024_R10m_E32N18_03035_V01_R00/CLMS_HRLVLCC_CTY_S2024_R10m_E32N18_03035_V01_R00.tif.aux.xml', '/content/drive/Shareddrives/Progetti/Progetto_MLDM/data/raw/zone_test/albacete_spagna/2024/CLMS_HRLVLCC_CTY_S2024_R10m_E32N18_03035_V01_R00/CLMS_HRLVLCC_CTY_S2024_R10m_E32N18_03035_V01_R00.xml'], 'bbox': [-2.9952, 38.4339, -1.6912, 39.4732]}, 'jerez_spagna': {'tif': ['/content/drive/Shareddrives/Proge

In [6]:
import rasterio
from src.config import CAPITANATA_BBOX, VICINO_CAPITANATA_BBOX, YEARS_TO_FETCH
from src.sentinel2_downloader import download_area

for zone_name, file_paths in tifs_3035.items():
    if not isinstance(file_paths, dict) or "bbox" not in file_paths:
        continue

    bbox = file_paths["bbox"]

    out_directory = DATA_DIR / "processed" / "zone_test" / f"sentinel2_{zone_name}_area"

    # avvio del download
    downloaded_files = download_area(
        name=zone_name,
        bbox=bbox,
        years=["2024"],
        months=range(1, 13),
        out_dir=str(out_directory)
    )

    # riepilogo dei dati scaricati
    tif_files = sorted(list(out_directory.glob("*.tif")))

    print("\n" + "-" * 50)
    print(f"Riepilogo dati satellitari zona {zone_name}")
    print("-" * 50)
    print(f"Cartella: {out_directory.resolve()}")
    print(f"File GeoTIFF mensili trovati: {len(tif_files)} (attesi: {len(YEARS_TO_FETCH) * 12})")

    total_size_mb = 0.0
    for f in tif_files:
        size_mb = f.stat().st_size / (1024 * 1024)
        total_size_mb += size_mb
        print(f"  • {f.name} ({size_mb:.1f} MB)")

    print(f"Spazio totale occupato: {total_size_mb / 1024:.2f} GB")

    # controllo di integrità sul primo file disponibile
    if tif_files:                                                
        with rasterio.open(tif_files[0]) as src:                 
            print(f"\nVerifica formato raster ({tif_files[0].name}):")                                                      
            print(f"  • Dimensioni: {src.width} x {src.height} pixel")                                                        
            print(f"  • Numero bande: {src.count}") 

Inizio download area 'albacete_spagna' per l'anno 2024
Mese 01 già presente e completo: /content/drive/Shareddrives/Progetti/Progetto_MLDM/data/processed/zone_test/sentinel2_albacete_spagna_area/albacete_spagna_2024_01.tif (925.9 MB)
Mese 02 già presente e completo: /content/drive/Shareddrives/Progetti/Progetto_MLDM/data/processed/zone_test/sentinel2_albacete_spagna_area/albacete_spagna_2024_02.tif (938.2 MB)
Mese 03 già presente e completo: /content/drive/Shareddrives/Progetti/Progetto_MLDM/data/processed/zone_test/sentinel2_albacete_spagna_area/albacete_spagna_2024_03.tif (932.3 MB)
Mese 04 già presente e completo: /content/drive/Shareddrives/Progetti/Progetto_MLDM/data/processed/zone_test/sentinel2_albacete_spagna_area/albacete_spagna_2024_04.tif (942.2 MB)
Mese 05 già presente e completo: /content/drive/Shareddrives/Progetti/Progetto_MLDM/data/processed/zone_test/sentinel2_albacete_spagna_area/albacete_spagna_2024_05.tif (929.3 MB)
Mese 06 già presente e completo: /content/drive/Sh

In [7]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import rasterio
from rasterio.enums import Resampling
from rasterio.merge import merge
from rasterio.transform import xy
from rasterio.warp import transform
from tqdm import tqdm

from src.config import EXCLUDED_CROP_CLASSES
from src.feature_engineering import (
    FEATURE_LIST,
    extract_features_from_monthly_stack,
)

# ==============================================================================
# PARAMETRI DI CONFIGURAZIONE
# ==============================================================================
TARGET_YEAR = 2024
N_SAMPLES_PER_ZONE = 100_000
RANDOM_SEED = 42
MIN_VALID_MONTHS = 6  # minimo mesi con dati validi richiesti per punto
FORCE_RECREATE = False  # se False, salta le zone già elaborate se il file esiste

# Cartella di output in cui salvare i 4 dataset
datasets_output_dir = DATA_DIR / "processed" / "zone_test" / "datasets"
datasets_output_dir.mkdir(parents=True, exist_ok=True)

# Caricamento eventuale legenda ufficiale
legend = {}
legend_path = DATA_DIR / "processed" / "legend.json"
if legend_path.exists():
    with open(legend_path, "r", encoding="utf-8") as f:
        legend = json.load(f)

# Dizionario per memorizzare i DataFrame in memoria
datasets_test = {}

# Barra di avanzamento principale sulle 4 zone
pbar_zones = tqdm(tifs_4326.items(), desc="Elaborazione dataset zone di test", unit="zona")

for zone_name, gt_files in pbar_zones:
    pbar_zones.set_postfix_str(f"Zona: {zone_name}")
    
    parquet_path = datasets_output_dir / f"dataset_{zone_name}.parquet"
    csv_path = datasets_output_dir / f"dataset_{zone_name}.csv"

    # 1. Verifica se il dataset è già stato generato
    if not FORCE_RECREATE and parquet_path.exists():
        tqdm.write(f"[{zone_name}] Dataset già presente su disco ({parquet_path.name}). Caricamento da cache...")
        datasets_test[zone_name] = pd.read_parquet(parquet_path)
        continue

    # 2. Controllo presenza raster Sentinel-2 per tutti i 12 mesi
    sentinel_dir = DATA_DIR / "processed" / "zone_test" / f"sentinel2_{zone_name}_area"
    if not sentinel_dir.exists():
        tqdm.write(f"[{zone_name}] Cartella Sentinel non trovata ({sentinel_dir}). Salto.")
        continue

    sentinel_monthly_files = {}
    for m in range(1, 13):
        candidates = sorted(list(sentinel_dir.glob(f"*{TARGET_YEAR}*_{m:02d}.tif")))
        if candidates:
            sentinel_monthly_files[m] = candidates[0]

    if len(sentinel_monthly_files) < 12:
        missing = [m for m in range(1, 13) if m not in sentinel_monthly_files]
        tqdm.write(f"[{zone_name}] Mancano i file Sentinel per i mesi: {missing} ({len(sentinel_monthly_files)}/12 scaricati). Salto per ora.")
        continue

    # 3. Lettura Ground Truth Copernicus ed individuazione campi agricoli
    tqdm.write(f"[{zone_name}] Lettura mappa del suolo (Copernicus CLMS)...")
    if len(gt_files) > 1:
        src_list = [rasterio.open(f) for f in gt_files]
        gt_mosaic, gt_transform = merge(src_list, resampling=Resampling.nearest)
        gt_crs = src_list[0].crs
        for s in src_list:
            s.close()
        gt_data = gt_mosaic.squeeze()
    else:
        with rasterio.open(gt_files[0]) as s:
            gt_data = s.read(1)
            gt_transform = s.transform
            gt_crs = s.crs

    # Filtro pixel agricoli validi (esclusione classi non agricole / incerte)
    valid_mask = ~np.isin(gt_data, EXCLUDED_CROP_CLASSES) & (gt_data > 0)
    valid_rows, valid_cols = np.where(valid_mask)
    total_valid = len(valid_rows)

    if total_valid == 0:
        tqdm.write(f"[{zone_name}] Nessun pixel agricolo valido trovato. Salto.")
        continue

    # 4. Campionamento casuale uniforme (mantiene la reale proporzione naturale delle colture)
    n_samples = min(N_SAMPLES_PER_ZONE, total_valid)
    tqdm.write(f"[{zone_name}] Campi agricoli identificati: {total_valid:,}. Campionamento di {n_samples:,} punti a 10m...")

    np.random.seed(RANDOM_SEED)
    chosen_indices = np.random.choice(total_valid, size=n_samples, replace=False)
    sample_rows = valid_rows[chosen_indices]
    sample_cols = valid_cols[chosen_indices]
    sample_codes = gt_data[sample_rows, sample_cols].astype(int)

    # Coordinate geografiche native dei punti estratti
    sample_xs, sample_ys = xy(gt_transform, sample_rows, sample_cols)
    sample_lons = np.array(sample_xs)
    sample_lats = np.array(sample_ys)

    # 5. Trasformazione coordinate nel CRS dei raster Sentinel-2
    with rasterio.open(sentinel_monthly_files[1]) as ref_s:
        sentinel_crs = ref_s.crs

    sentinel_xs, sentinel_ys = transform(gt_crs, sentinel_crs, sample_lons, sample_lats)
    sentinel_coords = list(zip(sentinel_xs, sentinel_ys))

    # 6. Campionamento spettrale a risoluzione pura 10m x 10m sui 12 mesi
    monthly_points = np.zeros((12, 6, n_samples), dtype=np.float32)
    pbar_months = tqdm(range(1, 13), desc=f"[{zone_name}] Campionamento 12 mesi (10m)", leave=False)

    for m in pbar_months:
        with rasterio.open(sentinel_monthly_files[m]) as src:
            # rasterio.sample estrae i pixel esatti 10x10m alle coordinate richieste in C
            samples = np.array(list(src.sample(sentinel_coords)), dtype=np.float32)
            monthly_points[m - 1, :, :] = samples.T

    # 7. Filtro di qualità: scarto punti con osservazioni insufficienti (< 6 mesi)
    # B04 (Rosso, indice 2) > 0 indica osservazione valida
    valid_months_per_point = np.sum(monthly_points[:, 2, :] > 0, axis=0)
    quality_mask = valid_months_per_point >= MIN_VALID_MONTHS
    n_passed = int(np.sum(quality_mask))

    if n_passed < n_samples:
        tqdm.write(f"[{zone_name}] Scartati {n_samples - n_passed:,} punti con < {MIN_VALID_MONTHS} mesi validi.")
        monthly_points = monthly_points[:, :, quality_mask]
        sample_codes = sample_codes[quality_mask]
        sample_lons = sample_lons[quality_mask]
        sample_lats = sample_lats[quality_mask]

    # 8. Calcolo vettoriale delle 51 feature fenologiche
    tqdm.write(f"[{zone_name}] Calcolo delle 51 feature fenologiche su {n_passed:,} punti...")
    _, features_matrix = extract_features_from_monthly_stack(monthly_points)

    # 9. Creazione DataFrame
    df_zone = pd.DataFrame(features_matrix, columns=FEATURE_LIST)
    df_zone["code"] = sample_codes
    if legend:
        df_zone["crop_name"] = df_zone["code"].astype(str).map(legend)
    df_zone["lon"] = sample_lons
    df_zone["lat"] = sample_lats
    df_zone["zone"] = zone_name
    df_zone["year"] = TARGET_YEAR

    datasets_test[zone_name] = df_zone

    # 10. Salvataggio su disco (Parquet e CSV)
    tqdm.write(f"[{zone_name}] Salvataggio dataset su disco...")
    df_zone.to_parquet(parquet_path, index=False)
    df_zone.to_csv(csv_path, index=False)
    tqdm.write(f"[{zone_name}] ✅ Dataset salvato con successo: {parquet_path.name} ({len(df_zone):,} righe)")

print("\n" + "=" * 60)
print("Elaborazione completata per tutte le zone disponibili!")
print("=" * 60)
for z_name, df_z in datasets_test.items():
    print(f"• {z_name}: {len(df_z):,} esempi, {df_z['code'].nunique()} classi colturali distinte")


Elaborazione dataset zone di test:   0%|          | 0/4 [00:00<?, ?zona/s, Zona: albacete_spagna]

[albacete_spagna] Dataset già presente su disco (dataset_albacete_spagna.parquet). Caricamento da cache...


Elaborazione dataset zone di test:  25%|██▌       | 1/4 [00:02<00:07,  2.44s/zona, Zona: jerez_spagna]   

[jerez_spagna] Dataset già presente su disco (dataset_jerez_spagna.parquet). Caricamento da cache...


Elaborazione dataset zone di test:  50%|█████     | 2/4 [00:02<00:02,  1.19s/zona, Zona: rovato]      

[rovato] Dataset già presente su disco (dataset_rovato.parquet). Caricamento da cache...


Elaborazione dataset zone di test:  75%|███████▌  | 3/4 [00:03<00:00,  1.27zona/s, Zona: vicino_capitanata]

[vicino_capitanata] Dataset già presente su disco (dataset_vicino_capitanata.parquet). Caricamento da cache...


Elaborazione dataset zone di test: 100%|██████████| 4/4 [00:03<00:00,  1.22zona/s, Zona: vicino_capitanata]


Elaborazione completata per tutte le zone disponibili!
• albacete_spagna: 100,000 esempi, 14 classi colturali distinte
• jerez_spagna: 100,000 esempi, 16 classi colturali distinte
• rovato: 100,000 esempi, 17 classi colturali distinte
• vicino_capitanata: 100,000 esempi, 15 classi colturali distinte


### Test - Vicino Capitanata


In [ ]:
import json
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML
from sklearn.metrics import classification_report, accuracy_score, f1_score

from src.feature_engineering import FEATURE_LIST

# ==============================================================================
# 1. CARICAMENTO DATASET VICINO CAPITANATA
# ==============================================================================
zone_target = "vicino_capitanata"
dataset_path = DATA_DIR / "processed" / "zone_test" / "datasets" / f"dataset_{zone_target}.parquet"

if zone_target in datasets_test:
    df_test = datasets_test[zone_target]
    print(f"Dataset '{zone_target}' prelevato dalla memoria: {len(df_test):,} righe")
elif dataset_path.exists():
    df_test = pd.read_parquet(dataset_path)
    print(f"Dataset '{zone_target}' caricato da disco: {len(df_test):,} righe")
else:
    raise FileNotFoundError(f"Dataset non trovato in {dataset_path}. Assicurati di aver eseguito la cella di estrazione.")

X_test = df_test[FEATURE_LIST]
y_test = df_test["code"].astype(int)

# Caricamento legenda colture
legend = {}
legend_path = DATA_DIR / "processed" / "legend.json"
if legend_path.exists():
    with open(legend_path, "r", encoding="utf-8") as f:
        legend = json.load(f)

# ==============================================================================
# 2. CARICAMENTO DEI MODELLI ADDESTRATI
# ==============================================================================
MODELS_DIR = DATA_DIR / "models"

def load_model(name, filename):
    candidates = [
        MODELS_DIR / name / filename,
        MODELS_DIR / filename
    ]
    for p in candidates:
        if p.exists():
            print(f"✓ Caricato {name} da: {p.relative_to(DATA_DIR)}")
            return joblib.load(p)
    print(f"✗ Modello {name} non ancora trovato su disco ({filename}).")
    return None

models_to_test = {
    "Random Forest": load_model("random_forest", "random_forest_crop_model.joblib"),
    "Extra Trees": load_model("extra_trees", "extra_trees_crop_model.joblib"),
    "XGBoost": load_model("xgboost", "xgboost_crop_model.joblib")
}

def predict_model(payload, X):
    model = payload["model"]
    preds = model.predict(X)
    if "label_encoder" in payload and payload["label_encoder"] is not None:
        preds = payload["label_encoder"].inverse_transform(preds)
    return preds

# ==============================================================================
# 3. PREDIZIONE E GENERAZIONE DEI REPORT
# ==============================================================================
unique_classes = sorted(y_test.unique())
target_names = [f"{legend.get(str(c), c)} ({c})" for c in unique_classes]

reports_dict = {}
summary_metrics = []

for m_name, payload in models_to_test.items():
    if payload is None:
        continue
    
    preds = predict_model(payload, X_test)
    
    acc = accuracy_score(y_test, preds)
    f1_m = f1_score(y_test, preds, average="macro", zero_division=0)
    f1_w = f1_score(y_test, preds, average="weighted", zero_division=0)
    
    summary_metrics.append({
        "Modello": m_name,
        "Accuracy (%)": round(acc * 100, 2),
        "Macro F1 (%)": round(f1_m * 100, 2),
        "Weighted F1 (%)": round(f1_w * 100, 2)
    })
    
    rep = classification_report(
        y_test, preds, labels=unique_classes, target_names=target_names,
        output_dict=True, zero_division=0
    )
    df_rep = pd.DataFrame(rep).T.round(3)
    df_rep["support"] = df_rep["support"].astype(int)
    reports_dict[m_name] = df_rep

# ==============================================================================
# 4. STAMPA DEI REPORT UNO ACCANTO ALL'ALTRO (HTML)
# ==============================================================================
print("\n" + "=" * 90)
print(f"CONFRONTO MODELLI SU: {zone_target.upper()} ({len(df_test):,} campioni)")
print("=" * 90)

# 4.1 Tabella riepilogativa globale
df_summary = pd.DataFrame(summary_metrics)
display(df_summary)

# 4.2 Report dettagliati affiancati tramite HTML Flexbox
html_cards = []
for m_name, df_rep in reports_dict.items():
    table_html = df_rep.to_html(classes="table", justify="center")
    card_html = f"""
    <div style="flex: 1; min-width: 380px; margin: 10px; border: 1px solid #d0d7de; border-radius: 8px; padding: 12px; background-color: #fafbfc; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
        <h3 style="text-align: center; color: #1f4287; margin-top: 0; padding-bottom: 6px; border-bottom: 2px solid #1f4287;">{m_name}</h3>
        <div style="overflow-x: auto; max-height: 480px;">
            {table_html}
        </div>
    </div>
    """
    html_cards.append(card_html)

full_html = f"""
<div style="display: flex; flex-direction: row; flex-wrap: nowrap; overflow-x: auto; width: 100%; gap: 10px;">
    {''.join(html_cards)}
</div>
"""
display(HTML(full_html))

# ==============================================================================
# 5. TABELLA COMPARATIVA DIRETTA F1-SCORE PER COLTURA
# ==============================================================================
if reports_dict:
    comp_f1_data = {}
    for m_name, df_rep in reports_dict.items():
        comp_f1_data[f"F1_{m_name}"] = df_rep["f1-score"]
    
    comp_f1_df = pd.DataFrame(comp_f1_data)
    comp_f1_df["Support"] = next(iter(reports_dict.values()))["support"]
    
    print("\nConfronto sintetico F1-Score per classe colturale:")
    display(comp_f1_df)

    # ==============================================================================
    # 6. GRAFICO A BARRE COMPARATIVO F1-SCORE PER CLASSE
    # ==============================================================================
    crops_only_df = comp_f1_df.drop(index=["accuracy", "macro avg", "weighted avg"], errors="ignore")
    
    fig, ax = plt.subplots(figsize=(14, max(6, len(crops_only_df) * 0.45)), dpi=100)
    f1_cols = [c for c in crops_only_df.columns if c.startswith("F1_")]
    crops_only_df[f1_cols].plot(kind="barh", ax=ax, width=0.8, colormap="viridis", edgecolor="black")
    
    ax.set_title(f"Confronto F1-Score per Classe - Zona: {zone_target}", fontsize=14, fontweight="bold", pad=15)
    ax.set_xlabel("F1-Score", fontsize=12)
    ax.set_xlim(0, 1.05)
    ax.grid(axis="x", linestyle="--", alpha=0.7)
    ax.legend([c.replace("F1_", "") for c in f1_cols], loc="lower right")
    plt.tight_layout()
    plt.show()
